<a href="https://colab.research.google.com/github/UlisesDev-10/Sistemas-Operativos/blob/main/Mi_Copia_de_Sistemas_Operativos_U2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💻 Sistemas Operativos - Administración de Procesos - Jearaquía, Implantación, IPC
**Asignatura:** Fundamentos y Diseño de Sistemas Operativos (6.º Semestre)    
**Unidad II:** Administración de Procesos  
**Tiempo estimado:** 2 horas (Asíncrono)  
**Repositorio Base:** Integración con GitHub  

---

## 1. Objetivos de Aprendizaje
1. Examinar la tabla de procesos del sistema operativo huésped utilizando comandos nativos de Linux (`ps`) en Google Colab para identificar atributos clave como identificadores de procesos (**PID**/**PPID**) y relaciones jerárquicas.

2. Explorar la estructura de datos interna que el kernel de Linux expone a través del sistema de archivos virtual (`/proc`) para comprender cómo se recopila y expone la información de ejecución a nivel de sistema.

3. Analizar la jerarquía de procesos y las relaciones de descendencia en un entorno Linux real empleando herramientas como pstree y el sistema de archivos virtual `/proc` en Google Colab.

4. Implementar las llamadas al sistema clásicas de control de procesos (`fork`, `exec`, `exit`, `wait`) mediante un simulador orientado a objetos en C++17.

5. Examinar fenómenos fundamentales de los sistemas operativos, tales como la persistencia de metadatos en estados Zombie y la adopción automática de procesos huérfanos por parte del proceso raíz (`init`).

6. Examinar los recursos de IPC activos en el sistema operativo huésped (**memoria compartida, semáforos y colas de mensajes**) mediante el comando `ipcs` y parámetros del kernel en `/proc`.

7. Implementar el control de concurrencia y la exclusión mutua aplicando **semáforos** ***POSIX*** (`sem_t`, `sem_wait`, `sem_post`) para resolver el problema clásico del **Productor-Consumidor** con un buffer circular.

8. Desarrollar canales de comunicación unidireccional entre procesos independientes utilizando las llamadas al sistema `pipe()`, `fork()`, `read()` y `write()`.

---

## 2. Sección Práctica 1: Inspección del Sistema Operativo.

En esta sección analizarás los fundamentos de la administración de procesos en los sistemas operativos modernos mediante la inspección de la tabla de procesos en un entorno Linux real(Servidor Linux provisto por Google).

### [CÓDIGO 2.1] inspeccionar la tabla de procesos real del sistema operativo huésped
```bash
# 1. Visualizar la lista completa de procesos activos en formato detallado
!ps -aux

# 2. Inspeccionar la jerarquía de procesos y ver el proceso PID 1 (init/systemd)
!ps -ef --forest

# 3. Consultar la estructura de datos interna que Linux expone en el sistema de archivos /proc para un PID
!ls -l /proc/1/
!cat /proc/1/status | head -n 15
```

In [ ]:
# 1. Visualizar la lista completa de procesos activos en formato detallado
!ps -aux

# 2. Inspeccionar la jerarquía de procesos y ver el proceso PID 1 (init/systemd)
!ps -ef --forest

# 3. Consultar la estructura de datos interna que Linux expone en el sistema de archivos /proc para un PID
!ls -l /proc/1/
!cat /proc/1/status | head -n 15

USER         PID %CPU %MEM    VSZ   RSS TTY      STAT START   TIME COMMAND
root           1  0.0  0.0    984   560 ?        Ss   03:21   0:00 /sbin/docker-
root           7  0.5  0.5 1258724 71064 ?       Sl   03:21   0:02 /tools/node/b
root           9  0.0  0.0   7344  3580 ?        S    03:21   0:00 /bin/bash -e 
root          11  0.0  0.0   7344  1968 ?        S    03:21   0:00 /bin/bash -e 
root          12  0.0  0.2 1286680 34036 ?       Sl   03:21   0:00 /usr/colab/bi
root          41  0.0  0.0   5720  2212 ?        Ss   03:21   0:00 tail -n +0 -F
root          47  0.0  0.0   5720  2276 ?        Ss   03:21   0:00 tail -n +0 -F
root          69  5.3  0.0      0     0 ?        Z    03:21   0:21 [python3] <de
root          70  0.8  0.5 100940 77768 ?        S    03:21   0:03 python3 /usr/
root          83  0.0  0.0   7456  2032 ?        S    03:21   0:00 /bin/bash -e 
root          84  0.0  0.0  12024  7812 ?        S    03:21   0:00 sshd: /usr/sb
root         100  2.0  1.2 665196 

### [CÓDIGO 2.2] Implementación de la Estructura struct BCP en C++17
```cpp
%%writefile bcp_struct.cpp
// ==============================================================================
// EJERCICIO PRÁCTICO: ESTRUCTURA DEL BLOQUE DE CONTROL DE PROCESO (BCP)
// Lenguaje: C++17
// Asignatura: Sistemas Operativos - Unidad II: Administración de Procesos
// ==============================================================================

#include <iostream>
#include <string>
#include <vector>
#include <iomanip>
#include <cstdint>

// Enumeración de los Estados Conceptuales de un Proceso (Modelo de 5 Estados)
enum class EstadoProceso {
    NUEVO,
    LISTO,
    EJECUTANDO,
    BLOQUEADO,
    SALIENTE
};

// Función auxiliar para convertir el Enum a String legible
std::string estadoToString(EstadoProceso estado) {
    switch (estado) {
        case EstadoProceso::NUEVO:      return "NUEVO";
        case EstadoProceso::LISTO:      return "LISTO";
        case EstadoProceso::EJECUTANDO: return "EJECUTANDO";
        case EstadoProceso::BLOQUEADO:  return "BLOQUEADO";
        case EstadoProceso::SALIENTE:   return "SALIENTE";
        default:                        return "DESCONOCIDO";
    }
}

// Estructura de Datos del BCP (Process Control Block)
struct BCP {
    // 1. Identificadores
    uint32_t pid;                   // Process ID único
    uint32_t ppid;                  // Parent Process ID
    std::string nombre_programa;    // Nombre del ejecutable en disco

    // 2. Información de Estado del Procesador (Contexto Guardado)
    EstadoProceso estado_actual;    // Estado conceptual del proceso
    uint16_t PC;                    // Program Counter (Contador de Programa)
    uint16_t SP;                    // Stack Pointer (Puntero de Pila)
    int AC;                         // Acumulador
    uint8_t PSW_modo;               // 0 = Modo Usuario, 1 = Modo Kernel

    // 3. Información de Gestión de Memoria y Planificación
    uint32_t direccion_base_ram;    // Dirección inicial de memoria asignada
    uint32_t limite_memoria_bytes;  // Tamaño del espacio de direcciones
    int prioridad;                  // Nivel de prioridad de planificación

    // Método para imprimir la ficha técnica del BCP
    void mostrarBCP() const {
        std::cout << "┌" << std::string(65, '-') << "┐\n";
        std::cout << "│ FICHA TÉCNICA DEL BCP (PID: " << std::setw(5) << pid << ") - Program: "
                  << std::left << std::setw(20) << nombre_programa << "│\n";
        std::cout << "├" << std::string(65, '-') << "┤\n";
        std::cout << "│ Estado Actual  : " << std::setw(15) << estadoToString(estado_actual)
                  << " | PPID Padre     : " << std::setw(10) << ppid << " │\n";
        std::cout << "│ Contexto CPU   : PC=0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << PC << std::dec
                  << " | SP=0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << SP << std::dec
                  << " | AC=" << std::setw(6) << std::setfill(' ') << AC << " │\n";
        std::cout << "│ Modo PSW       : " << (PSW_modo == 1 ? "KERNEL (1)    " : "USUARIO (0)   ")
                  << " | Prioridad      : " << std::setw(10) << prioridad << " │\n";
        std::cout << "│ Memoria Asign. : Base=0x" << std::hex << std::uppercase << std::setw(6) << std::setfill('0') << direccion_base_ram << std::dec
                  << " | Límite=" << std::setw(6) << limite_memoria_bytes << " Bytes  │\n";
        std::cout << "└" << std::string(65, '-') << "┘\n";
    }
};

int main() {
    std::cout << "=================================================================\n";
    std::cout << "   SIMULACIÓN DE LA TABLA DE PROCESOS DEL SISTEMA OPERATIVO     \n";
    std::cout << "=================================================================\n\n";

    // Creación de una Tabla de Procesos simulada (Vector de BCPs)
    std::vector<BCP> tabla_procesos;

    // Inicialización del Proceso 1 (Navegador)
    BCP p1;
    p1.pid = 101;
    p1.ppid = 1; // Hijo del proceso init
    p1.nombre_programa = "chrome.exe";
    p1.estado_actual = EstadoProceso::EJECUTANDO;
    p1.PC = 0x0420;
    p1.SP = 0x0FFF;
    p1.AC = 150;
    p1.PSW_modo = 0; // Modo Usuario
    p1.direccion_base_ram = 0x10000;
    p1.limite_memoria_bytes = 65536;
    p1.prioridad = 5;

    // Inicialización del Proceso 2 (Compilador Bloqueado por E/S)
    BCP p2;
    p2.pid = 102;
    p2.ppid = 101;
    p2.nombre_programa = "g++_compiler";
    p2.estado_actual = EstadoProceso::BLOQUEADO;
    p2.PC = 0x08A2;
    p2.SP = 0x0E00;
    p2.AC = 0;
    p2.PSW_modo = 0; // Modo Usuario
    p2.direccion_base_ram = 0x20000;
    p2.limite_memoria_bytes = 131072;
    p2.prioridad = 10;

    // Agregar procesos a la tabla
    tabla_procesos.push_back(p1);
    tabla_procesos.push_back(p2);

    // Desplegar la información guardada en los BCPs
    for (const auto& bcp : tabla_procesos) {
        bcp.mostrarBCP();
        std::cout << "\n";
    }

    return 0;
}
```

In [ ]:
%%writefile bcp_struct.cpp
// ==============================================================================
// EJERCICIO PRÁCTICO: ESTRUCTURA DEL BLOQUE DE CONTROL DE PROCESO (BCP)
// Lenguaje: C++17
// Asignatura: Sistemas Operativos - Unidad II: Administración de Procesos
// ==============================================================================

# include <iostream>
# include <string>
# include <vector>
# include <iomanip>
# include <cstdint>

// Enumeración de los Estados Conceptuales de un Proceso (Modelo de 5 Estados)
enum class EstadoProceso {
    NUEVO,
    LISTO,
    EJECUTANDO,
    BLOQUEADO,
    SALIENTE
};

// Función auxiliar para convertir el Enum a String legible
std::string estadoToString(EstadoProceso estado) {
    switch (estado) {
        case EstadoProceso::NUEVO:      return "NUEVO";
        case EstadoProceso::LISTO:      return "LISTO";
        case EstadoProceso::EJECUTANDO: return "EJECUTANDO";
        case EstadoProceso::BLOQUEADO:  return "BLOQUEADO";
        case EstadoProceso::SALIENTE:   return "SALIENTE";
        default:                        return "DESCONOCIDO";
    }
}

// Estructura de Datos del BCP (Process Control Block)
struct BCP {
    // 1. Identificadores
    uint32_t pid;                   // Process ID único
    uint32_t ppid;                  // Parent Process ID
    std::string nombre_programa;    // Nombre del ejecutable en disco

    // 2. Información de Estado del Procesador (Contexto Guardado)
    EstadoProceso estado_actual;    // Estado conceptual del proceso
    uint16_t PC;                    // Program Counter (Contador de Programa)
    uint16_t SP;                    // Stack Pointer (Puntero de Pila)
    int AC;                         // Acumulador
    uint8_t PSW_modo;               // 0 = Modo Usuario, 1 = Modo Kernel

    // 3. Información de Gestión de Memoria y Planificación
    uint32_t direccion_base_ram;    // Dirección inicial de memoria asignada
    uint32_t limite_memoria_bytes;  // Tamaño del espacio de direcciones
    int prioridad;                  // Nivel de prioridad de planificación

    // Método para imprimir la ficha técnica del BCP
    void mostrarBCP() const {
        std::cout << "┌" << std::string(65, '-') << "┐\n";
        std::cout << "│ FICHA TÉCNICA DEL BCP (PID: " << std::setw(5) << pid << ") - Program: "
                  << std::left << std::setw(20) << nombre_programa << "│\n";
        std::cout << "├" << std::string(65, '-') << "┤\n";
        std::cout << "│ Estado Actual  : " << std::setw(15) << estadoToString(estado_actual)
                  << " | PPID Padre     : " << std::setw(10) << ppid << " │\n";
        std::cout << "│ Contexto CPU   : PC=0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << PC << std::dec
                  << " | SP=0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << SP << std::dec
                  << " | AC=" << std::setw(6) << std::setfill(' ') << AC << " │\n";
        std::cout << "│ Modo PSW       : " << (PSW_modo == 1 ? "KERNEL (1)    " : "USUARIO (0)   ")
                  << " | Prioridad      : " << std::setw(10) << prioridad << " │\n";
        std::cout << "│ Memoria Asign. : Base=0x" << std::hex << std::uppercase << std::setw(6) << std::setfill('0') << direccion_base_ram << std::dec
                  << " | Límite=" << std::setw(6) << limite_memoria_bytes << " Bytes  │\n";
        std::cout << "└" << std::string(65, '-') << "┘\n";
    }
};

int main() {
    std::cout << "=================================================================\n";
    std::cout << "   SIMULACIÓN DE LA TABLA DE PROCESOS DEL SISTEMA OPERATIVO     \n";
    std::cout << "=================================================================\n\n";

    // Creación de una Tabla de Procesos simulada (Vector de BCPs)
    std::vector<BCP> tabla_procesos;

    // Inicialización del Proceso 1 (Navegador)
    BCP p1;
    p1.pid = 101;
    p1.ppid = 1; // Hijo del proceso init
    p1.nombre_programa = "chrome.exe";
    p1.estado_actual = EstadoProceso::EJECUTANDO;
    p1.PC = 0x0420;
    p1.SP = 0x0FFF;
    p1.AC = 150;
    p1.PSW_modo = 0; // Modo Usuario
    p1.direccion_base_ram = 0x10000;
    p1.limite_memoria_bytes = 65536;
    p1.prioridad = 5;

    // Inicialización del Proceso 2 (Compilador Bloqueado por E/S)
    BCP p2;
    p2.pid = 102;
    p2.ppid = 101;
    p2.nombre_programa = "g++_compiler";
    p2.estado_actual = EstadoProceso::BLOQUEADO;
    p2.PC = 0x08A2;
    p2.SP = 0x0E00;
    p2.AC = 0;
    p2.PSW_modo = 0; // Modo Usuario
    p2.direccion_base_ram = 0x20000;
    p2.limite_memoria_bytes = 131072;
    p2.prioridad = 10;

    // Agregar procesos a la tabla
    tabla_procesos.push_back(p1);
    tabla_procesos.push_back(p2);

    // Desplegar la información guardada en los BCPs
    for (const auto& bcp : tabla_procesos) {
        bcp.mostrarBCP();
        std::cout << "\n";
    }

    return 0;
}

Writing bcp_struct.cpp


### [CÓDIGO 2.3] Compilación y ejecución del código
```bash
# Compilar el código en C++17 con el máximo nivel de optimización (-O3)
!g++ -O3 -std=c++17 bcp_struct.cpp -o bcp_struct

# Ejecutar la simulación
!./bcp_struct
```

In [ ]:
# Compilar el código en C++17 con el máximo nivel de optimización (-O3)
!g++ -O3 -std=c++17 bcp_struct.cpp -o bcp_struct

# Ejecutar la simulación
!./bcp_struct

   SIMULACIÓN DE LA TABLA DE PROCESOS DEL SISTEMA OPERATIVO     

┌-----------------------------------------------------------------┐
│ FICHA TÉCNICA DEL BCP (PID:   101) - Program: chrome.exe          │
├-----------------------------------------------------------------┤
│ Estado Actual  : EJECUTANDO      | PPID Padre     : 1          │
│ Contexto CPU   : PC=0x4200 | SP=0xFFF0 | AC=150    │
│ Modo PSW       : USUARIO (0)    | Prioridad      : 5          │
│ Memoria Asign. : Base=0x100000 | Límite=655360 Bytes  │
└-----------------------------------------------------------------┘

┌-----------------------------------------------------------------┐
│ FICHA TÉCNICA DEL BCP (PID: 10200) - Program: g++_compiler00000000│
├-----------------------------------------------------------------┤
│ Estado Actual  : BLOQUEADO000000 | PPID Padre     : 1010000000 │
│ Contexto CPU   : PC=0x8A20 | SP=0xE000 | AC=0      │
│ Modo PSW       : USUARIO (0)    | Prioridad      : 10         │
│ Memoria Asign. : 

## 3. Sección Práctica 2: Jerarquía de Procesos.  

En esta sección explorarás y comprenderás la jerarquía, el ciclo de vida y la gestión de procesos en sistemas operativos mediante la inspección de estructuras del kernel en Linux y la simulación algorítmica de llamadas al sistema (`fork`, `exec`, `exit`, `wait`)

### [CÓDIGO 3.1] Inspección de la Jerarquía de Procesos en Linux.

Ejecutar los siguientes comandos en celdas de código bash en Google Colab:
```bash
# 1. Mostrar la jerarquía de procesos en formato de árbol (Tree)
!pstree -p

# 2. Consultar el BCP expuesto por Linux en /proc para el proceso actual
!cat /proc/self/status | grep -E "Name|State|Pid|PPid|Threads"

# 3. Identificar el proceso raíz init/systemd (PID 1)
!ps -p 1 -o pid,ppid,comm,state
```

In [ ]:
# 1. Mostrar la jerarquía de procesos en formato de árbol (Tree)
!pstree -p

# 2. Consultar el BCP expuesto por Linux en /proc para el proceso actual
!cat /proc/self/status | grep -E "Name|State|Pid|PPid|Threads"

# 3. Identificar el proceso raíz init/systemd (PID 1)
!ps -p 1 -o pid,ppid,comm,state

docker-init(1)─┬─node(7)─┬─colab-fileshim.(70)
               │         ├─jupyter-server(100)─┬─python3(1614)─┬─pstree(1887)
               │         │                     │               ├─{python3}(1622)
               │         │                     │               ├─{python3}(1623)
               │         │                     │               ├─{python3}(1624)
               │         │                     │               ├─{python3}(1625)
               │         │                     │               ├─{python3}(1626)
               │         │                     │               ├─{python3}(1627)
               │         │                     │               ├─{python3}(1628)
               │         │                     │               ├─{python3}(1629)
               │         │                     │               └─{python3}(1634)
               │         │                     ├─{jupyter-server}(118)
               │         │                     ├─{jupyter-server}(1611)
   

### [CÓDIGO 3.2] Inspección de la Jerarquía de Procesos en Linux.

Simulación de Jerarquía, BCP y Cambio de Contexto, copiar y ejecutar el siguiente código en Colab:

```cpp
%%writefile laboratorio_jerarquia_bcp.cpp
// ==============================================================================
// EJERCICIO PRÁCTICO: JERARQUÍA DE PROCESOS Y CAMBIO DE CONTEXTO
// Lenguaje: C++17
// Asignatura: Sistemas Operativos - Unidad II: Administración de Procesos
// ==============================================================================

#include <iostream>
#include <vector>
#include <string>
#include <iomanip>
#include <memory>

enum class Estado { NUEVO, LISTO, EJECUTANDO, BLOQUEADO, ZOMBIE, TERMINADO };

std::string estadoToString(Estado e) {
    switch (e) {
        case Estado::NUEVO:      return "NUEVO";
        case Estado::LISTO:      return "LISTO";
        case Estado::EJECUTANDO: return "EJECUTANDO";
        case Estado::BLOQUEADO:  return "BLOQUEADO";
        case Estado::ZOMBIE:     return "ZOMBIE";
        case Estado::TERMINADO:  return "TERMINADO";
        default:                 return "DESCONOCIDO";
    }
}

// Estructura del Bloque de Control de Proceso (BCP)
struct BCP {
    int pid;
    int ppid;
    std::string nombre_programa;
    Estado estado;
    uint16_t PC; // Program Counter
    uint16_t SP; // Stack Pointer
    int PSW_modo; // 0 = Usuario, 1 = Kernel
    int codigo_salida;

    void imprimir() const {
        std::cout << "[BCP PID=" << std::setw(3) << pid
                  << " | PPID=" << std::setw(3) << ppid
                  << " | Prog: " << std::left << std::setw(12) << nombre_programa
                  << " | Estado: " << std::setw(10) << estadoToString(estado)
                  << " | PC: 0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << PC << std::dec
                  << " | Modo: " << (PSW_modo == 1 ? "KERNEL" : "USUARIO") << "]\n";
    }
};

// Tabla de Procesos Simulada del Sistema Operativo
class SistemaOperativoSimulado {
private:
    std::vector<BCP> tabla_procesos;
    int contador_pid = 100;
    int pid_activo = -1;

public:
    SistemaOperativoSimulado() {
        // Crear proceso raíz (init - PID 1)
        BCP init{1, 0, "init", Estado::EJECUTANDO, 0x0010, 0x0FFF, 1, 0};
        tabla_procesos.push_back(init);
        pid_activo = 1;
    }

    // Simulación de la llamada fork()
    int simular_fork(int pid_padre, const std::string& nombre_hijo) {
        BCP* padre = obtenerBCP(pid_padre);
        if (!padre) return -1;

        contador_pid++;
        BCP hijo;
        hijo.pid = contador_pid;
        hijo.ppid = padre->pid;
        hijo.nombre_programa = nombre_hijo;
        hijo.estado = Estado::LISTO;
        hijo.PC = padre->PC; // Copia el PC del padre (misma instrucción)
        hijo.SP = padre->SP - 0x0100; // Nueva pila
        hijo.PSW_modo = 0;
        hijo.codigo_salida = 0;

        tabla_procesos.push_back(hijo);
        std::cout << "  [FORK] Proceso Padre PID " << pid_padre
                  << " creó al Proceso Hijo PID " << hijo.pid << " (" << nombre_hijo << ")\n";
        return hijo.pid;
    }

    // Simulación de la llamada exec()
    bool simular_exec(int pid, const std::string& nuevo_programa) {
        BCP* proc = obtenerBCP(pid);
        if (!proc) return false;

        std::cout << "  [EXEC] Proceso PID " << pid << " reemplaza su imagen por '" << nuevo_programa << "'\n";
        proc->nombre_programa = nuevo_programa;
        proc->PC = 0x1000; // Reinicia PC al punto de entrada del nuevo ejecutable
        return true;
    }

    // Simulación de terminación exit() y estado Zombie
    void simular_exit(int pid, int status) {
        BCP* proc = obtenerBCP(pid);
        if (!proc) return;

        proc->codigo_salida = status;
        proc->estado = Estado::ZOMBIE;
        std::cout << "  [EXIT] Proceso PID " << pid << " finalizó con código " << status
                  << " -> Entra a Estado ZOMBIE (esperando wait del padre)\n";
    }

    // Simulación de la llamada wait() recolectora
    void simular_wait(int pid_padre) {
        for (auto it = tabla_procesos.begin(); it != tabla_procesos.end(); ++it) {
            if (it->ppid == pid_padre && it->estado == Estado::ZOMBIE) {
                std::cout << "  [WAIT] Padre PID " << pid_padre
                          << " recolectó estado del Hijo ZOMBIE PID " << it->pid
                          << ". Eliminando BCP de la tabla.\n";
                tabla_procesos.erase(it);
                return;
            }
        }
    }

    // Simulación de adopción de huérfanos por init (PID 1)
    void atender_huerfanos(int pid_padre_muerto) {
        for (auto& proc : tabla_procesos) {
            if (proc.ppid == pid_padre_muerto) {
                std::cout << "  [ADOPCIÓN] Proceso Huérfano PID " << proc.pid
                          << " fue adoptado por init (PID 1)\n";
                proc.ppid = 1;
            }
        }
    }

    BCP* obtenerBCP(int pid) {
        for (auto& proc : tabla_procesos) {
            if (proc.pid == pid) return &proc;
        }
        return nullptr;
    }

    void mostrarTablaProcesos() const {
        std::cout << "\n=================================================================================\n";
        std::cout << "               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 \n";
        std::cout << "=================================================================================\n";
        for (const auto& proc : tabla_procesos) {
            proc.imprimir();
        }
        std::cout << "=================================================================================\n\n";
    }
};

int main() {
    SistemaOperativoSimulado so;
    so.mostrarTablaProcesos();

    std::cout << "--- 1. Creación de Jerarquía con fork() ---\n";
    int pid_bash = so.simular_fork(1, "bash");
    int pid_gcc  = so.simular_fork(pid_bash, "gcc");
    so.mostrarTablaProcesos();

    std::cout << "--- 2. Reemplazo de Imagen con exec() ---\n";
    so.simular_exec(pid_gcc, "cc1_compiler");
    so.mostrarTablaProcesos();

    std::cout << "--- 3. Generación de Proceso Zombie ---\n";
    so.simular_exit(pid_gcc, 0); // gcc termina pero bash no ha hecho wait
    so.mostrarTablaProcesos();

    std::cout << "--- 4. Recolección de Zombie con wait() ---\n";
    so.simular_wait(pid_bash);
    so.mostrarTablaProcesos();

    std::cout << "--- 5. Simulación de Proceso Huérfano ---\n";
    int pid_worker = so.simular_fork(pid_bash, "worker_task");
    so.simular_exit(pid_bash, 0); // Muere el padre bash antes que worker_task
    so.atender_huerfanos(pid_bash);
    so.mostrarTablaProcesos();

    return 0;
}
```

In [ ]:
%%writefile laboratorio_jerarquia_bcp.cpp
// ==============================================================================
// EJERCICIO PRÁCTICO: JERARQUÍA DE PROCESOS Y CAMBIO DE CONTEXTO
// Lenguaje: C++17
// Asignatura: Sistemas Operativos - Unidad II: Administración de Procesos
// ==============================================================================

# include <iostream>
# include <vector>
# include <string>
# include <iomanip>
# include <memory>

enum class Estado { NUEVO, LISTO, EJECUTANDO, BLOQUEADO, ZOMBIE, TERMINADO };

std::string estadoToString(Estado e) {
    switch (e) {
        case Estado::NUEVO:      return "NUEVO";
        case Estado::LISTO:      return "LISTO";
        case Estado::EJECUTANDO: return "EJECUTANDO";
        case Estado::BLOQUEADO:  return "BLOQUEADO";
        case Estado::ZOMBIE:     return "ZOMBIE";
        case Estado::TERMINADO:  return "TERMINADO";
        default:                 return "DESCONOCIDO";
    }
}

// Estructura del Bloque de Control de Proceso (BCP)
struct BCP {
    int pid;
    int ppid;
    std::string nombre_programa;
    Estado estado;
    uint16_t PC; // Program Counter
    uint16_t SP; // Stack Pointer
    int PSW_modo; // 0 = Usuario, 1 = Kernel
    int codigo_salida;

    void imprimir() const {
        std::cout << "[BCP PID=" << std::setw(3) << pid
                  << " | PPID=" << std::setw(3) << ppid
                  << " | Prog: " << std::left << std::setw(12) << nombre_programa
                  << " | Estado: " << std::setw(10) << estadoToString(estado)
                  << " | PC: 0x" << std::hex << std::uppercase << std::setw(4) << std::setfill('0') << PC << std::dec
                  << " | Modo: " << (PSW_modo == 1 ? "KERNEL" : "USUARIO") << "]\n";
    }
};

// Tabla de Procesos Simulada del Sistema Operativo
class SistemaOperativoSimulado {
private:
    std::vector<BCP> tabla_procesos;
    int contador_pid = 100;
    int pid_activo = -1;

public:
    SistemaOperativoSimulado() {
        // Crear proceso raíz (init - PID 1)
        BCP init{1, 0, "init", Estado::EJECUTANDO, 0x0010, 0x0FFF, 1, 0};
        tabla_procesos.push_back(init);
        pid_activo = 1;
    }

    // Simulación de la llamada fork()
    int simular_fork(int pid_padre, const std::string& nombre_hijo) {
        BCP* padre = obtenerBCP(pid_padre);
        if (!padre) return -1;

        contador_pid++;
        BCP hijo;
        hijo.pid = contador_pid;
        hijo.ppid = padre->pid;
        hijo.nombre_programa = nombre_hijo;
        hijo.estado = Estado::LISTO;
        hijo.PC = padre->PC; // Copia el PC del padre (misma instrucción)
        hijo.SP = padre->SP - 0x0100; // Nueva pila
        hijo.PSW_modo = 0;
        hijo.codigo_salida = 0;

        tabla_procesos.push_back(hijo);
        std::cout << "  [FORK] Proceso Padre PID " << pid_padre
                  << " creó al Proceso Hijo PID " << hijo.pid << " (" << nombre_hijo << ")\n";
        return hijo.pid;
    }

    // Simulación de la llamada exec()
    bool simular_exec(int pid, const std::string& nuevo_programa) {
        BCP* proc = obtenerBCP(pid);
        if (!proc) return false;

        std::cout << "  [EXEC] Proceso PID " << pid << " reemplaza su imagen por '" << nuevo_programa << "'\n";
        proc->nombre_programa = nuevo_programa;
        proc->PC = 0x1000; // Reinicia PC al punto de entrada del nuevo ejecutable
        return true;
    }

    // Simulación de terminación exit() y estado Zombie
    void simular_exit(int pid, int status) {
        BCP* proc = obtenerBCP(pid);
        if (!proc) return;

        proc->codigo_salida = status;
        proc->estado = Estado::ZOMBIE;
        std::cout << "  [EXIT] Proceso PID " << pid << " finalizó con código " << status
                  << " -> Entra a Estado ZOMBIE (esperando wait del padre)\n";
    }

    // Simulación de la llamada wait() recolectora
    void simular_wait(int pid_padre) {
        for (auto it = tabla_procesos.begin(); it != tabla_procesos.end(); ++it) {
            if (it->ppid == pid_padre && it->estado == Estado::ZOMBIE) {
                std::cout << "  [WAIT] Padre PID " << pid_padre
                          << " recolectó estado del Hijo ZOMBIE PID " << it->pid
                          << ". Eliminando BCP de la tabla.\n";
                tabla_procesos.erase(it);
                return;
            }
        }
    }

    // Simulación de adopción de huérfanos por init (PID 1)
    void atender_huerfanos(int pid_padre_muerto) {
        for (auto& proc : tabla_procesos) {
            if (proc.ppid == pid_padre_muerto) {
                std::cout << "  [ADOPCIÓN] Proceso Huérfano PID " << proc.pid
                          << " fue adoptado por init (PID 1)\n";
                proc.ppid = 1;
            }
        }
    }

    BCP* obtenerBCP(int pid) {
        for (auto& proc : tabla_procesos) {
            if (proc.pid == pid) return &proc;
        }
        return nullptr;
    }

    void mostrarTablaProcesos() const {
        std::cout << "\n=================================================================================\n";
        std::cout << "               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 \n";
        std::cout << "=================================================================================\n";
        for (const auto& proc : tabla_procesos) {
            proc.imprimir();
        }
        std::cout << "=================================================================================\n\n";
    }
};

int main() {
    SistemaOperativoSimulado so;
    so.mostrarTablaProcesos();

    std::cout << "--- 1. Creación de Jerarquía con fork() ---\n";
    int pid_bash = so.simular_fork(1, "bash");
    int pid_gcc  = so.simular_fork(pid_bash, "gcc");
    so.mostrarTablaProcesos();

    std::cout << "--- 2. Reemplazo de Imagen con exec() ---\n";
    so.simular_exec(pid_gcc, "cc1_compiler");
    so.mostrarTablaProcesos();

    std::cout << "--- 3. Generación de Proceso Zombie ---\n";
    so.simular_exit(pid_gcc, 0); // gcc termina pero bash no ha hecho wait
    so.mostrarTablaProcesos();

    std::cout << "--- 4. Recolección de Zombie con wait() ---\n";
    so.simular_wait(pid_bash);
    so.mostrarTablaProcesos();

    std::cout << "--- 5. Simulación de Proceso Huérfano ---\n";
    int pid_worker = so.simular_fork(pid_bash, "worker_task");
    so.simular_exit(pid_bash, 0); // Muere el padre bash antes que worker_task
    so.atender_huerfanos(pid_bash);
    so.mostrarTablaProcesos();

    return 0;
}

Writing laboratorio_jerarquia_bcp.cpp


### [CÓDIGO 3.3] Compilación y Ejecución del código
```c++
# Compilar en C++17 con g++
!g++ -O3 -std=c++17 laboratorio_jerarquia_bcp.cpp -o laboratorio_jerarquia_bcp

# Ejecutar la simulación
!./laboratorio_jerarquia_bcp
```

In [ ]:
# Compilar en C++17 con g++
!g++ -O3 -std=c++17 laboratorio_jerarquia_bcp.cpp -o laboratorio_jerarquia_bcp

# Ejecutar la simulación
!./laboratorio_jerarquia_bcp


               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 
[BCP PID=  1 | PPID=  0 | Prog: init         | Estado: EJECUTANDO | PC: 0x1000 | Modo: KERNEL]

--- 1. Creación de Jerarquía con fork() ---
  [FORK] Proceso Padre PID 1 creó al Proceso Hijo PID 101 (bash)
  [FORK] Proceso Padre PID 101 creó al Proceso Hijo PID 102 (gcc)

               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 
[BCP PID=100 | PPID=000 | Prog: init00000000 | Estado: EJECUTANDO | PC: 0x1000 | Modo: KERNEL]
[BCP PID=101 | PPID=100 | Prog: bash00000000 | Estado: LISTO00000 | PC: 0x1000 | Modo: USUARIO]
[BCP PID=102 | PPID=101 | Prog: gcc000000000 | Estado: LISTO00000 | PC: 0x1000 | Modo: USUARIO]

--- 2. Reemplazo de Imagen con exec() ---
  [EXEC] Proceso PID 102 reemplaza su imagen por 'cc1_compiler'

               ESTADO ACTUAL DE LA TABLA DE PROCESOS DEL SISTEMA                 
[BCP PID=100 | PPID=000 | Prog: init00000000 | Estado: EJECUTANDO | PC: 0x1000 | Modo:

## 4. Sección Práctica 3: Inter Process Communication (IPC).  

En esta sección implementarás y analizarás los mecanismos de Comunicación entre Procesos (IPC) y sincronización concurrente en sistemas operativos mediante la inspección de recursos del kernel en Linux y la resolución del problema clásico del Productor-Consumidor usando semáforos POSIX y tuberías en C++17.

### [CÓDIGO 4.1] Inspección de Mecanismos de IPC en el Entorno Linux.

Ejecutar los siguientes comandos en celdas de código bash para inspeccionar las facilidades de IPC del kernel Linux:
```bash
# 1. Consultar recursos de IPC activos en el sistema (Memoria compartida, Semáforos, Colas de mensajes)
!ipcs -a

# 2. Consultar el límite máximo del tamaño del buffer para tuberías en Linux (/proc)
!cat /proc/sys/fs/pipe-max-size

# 3. Crear y probar una tubería con nombre (FIFO) en la línea de comandos de Linux
!mkfifo /tmp/mi_tubería_fifo
!ls -l /tmp/mi_tubería_fifo
!rm /tmp/mi_tubería_fifo
```

In [ ]:
# 1. Consultar recursos de IPC activos en el sistema (Memoria compartida, Semáforos, Colas de mensajes)
!ipcs -a

# 2. Consultar el límite máximo del tamaño del buffer para tuberías en Linux (/proc)
!cat /proc/sys/fs/pipe-max-size

# 3. Crear y probar una tubería con nombre (FIFO) en la línea de comandos de Linux
!mkfifo /tmp/mi_tubería_fifo
!ls -l /tmp/mi_tubería_fifo
!rm /tmp/mi_tubería_fifo


------ Message Queues --------
key        msqid      owner      perms      used-bytes   messages    

------ Shared Memory Segments --------
key        shmid      owner      perms      bytes      nattch     status      

------ Semaphore Arrays --------
key        semid      owner      perms      nsems     

1048576
prw-r--r-- 1 root root 0 Nov 26 03:32 /tmp/mi_tubería_fifo


### [CÓDIGO 4.2] Simulación del Productor-Consumidor con Semáforos POSIX y Threads.

Copiar y ejecutar el siguiente código en Colab:
```cpp
%%writefile laboratorio_ipc_semaphoro.cpp
// ==============================================================================
// EJERCICIO PRÁCTICO: IPC Y SINCRONIZACIÓN CON SEMÁFOROS POSIX Y TUBERÍAS
// Lenguaje: C++17 (POSIX Threads & Semaphores)
// Asignatura: Sistemas Operativos - Unidad II: Administración de Procesos
// ==============================================================================

#include <iostream>
#include <thread>
#include <vector>
#include <chrono>
#include <semaphore.h>
#include <fcntl.h>
#include <unistd.h>
#include <sys/wait.h>

constexpr int TAMANO_BUFFER = 5;
constexpr int TOTAL_ITEMS = 10;

// Buffer circular compartido
int buffer[TAMANO_BUFFER];
int in_idx = 0;
int out_idx = 0;

// Semáforos POSIX
sem_t sem_huecos;    // Cuenta espacios libres en el buffer
sem_t sem_elementos; // Cuenta elementos disponibles para consumir
sem_t sem_mutex;     // Garantiza exclusión mutua (semáforo binario)

// Función del Proceso/Hilo Productor
void productor() {
    for (int i = 1; i <= TOTAL_ITEMS; ++i) {
        // 1. Producir dato
        int dato = i * 10;

        // 2. Esperar por un hueco libre
        sem_wait(&sem_huecos);

        // 3. Entrar a la región crítica (Mutex Down)
        sem_wait(&sem_mutex);

        // --- REGIÓN CRÍTICA ---
        buffer[in_idx] = dato;
        std::cout << "  [PRODUCTOR] Produjo e insertó item " << dato
                  << " en la posición " << in_idx << "\n";
        in_idx = (in_idx + 1) % TAMANO_BUFFER;
        // ----------------------

        // 4. Salir de la región crítica (Mutex Up)
        sem_post(&sem_mutex);

        // 5. Notificar que hay un nuevo elemento disponible
        sem_post(&sem_elementos);

        std::this_thread::sleep_for(std::chrono::milliseconds(150));
    }
}

// Función del Proceso/Hilo Consumidor
void consumidor() {
    for (int i = 1; i <= TOTAL_ITEMS; ++i) {
        // 1. Esperar a que exista al menos un elemento
        sem_wait(&sem_elementos);

        // 2. Entrar a la región crítica (Mutex Down)
        sem_wait(&sem_mutex);

        // --- REGIÓN CRÍTICA ---
        int dato = buffer[out_idx];
        std::cout << "    [CONSUMIDOR] Extrajo y consumió item " << dato
                  << " de la posición " << out_idx << "\n";
        out_idx = (out_idx + 1) % TAMANO_BUFFER;
        // ----------------------

        // 3. Salir de la región crítica (Mutex Up)
        sem_post(&sem_mutex);

        // 4. Notificar que se liberó un hueco
        sem_post(&sem_huecos);

        std::this_thread::sleep_for(std::chrono::milliseconds(300));
    }
}

// Función de demostración de Tubería Unidireccional (Pipe) entre procesos con fork()
void probar_tubería_pipe() {
    std::cout << "\n=================================================================\n";
    std::cout << "   DEMOSTRACIÓN DE COMUNICACIÓN POR TUBERÍA (PIPE EN LINUX)      \n";
    std::cout << "=================================================================\n";

    int fd[1]; // fd = lectura, fd[2] = escritura
    if (pipe(fd) == -1) {
        std::cerr << "Error al crear la tubería\n";
        return;
    }

    pid_t pid = fork();

    if (pid < 0) {
        std::cerr << "Error en fork()\n";
        return;
    }

    if (pid == 0) {
        // --- PROCESO HIJO (Escritor en la Tubería) ---
        close(fd); // Cierra el extremo de lectura que no utiliza
        std::string mensaje = "¡Saludos desde el Proceso Hijo mediante Pipe POSIX!";
        std::cout << "  [PROCESO HIJO] Enviando mensaje por el extremo de escritura fd[2]...\n";
        write(fd[2], mensaje.c_str(), mensaje.length() + 1);
        close(fd[2]); // Cierra escritura para enviar EOF
        _exit(0);
    } else {
        // --- PROCESO PADRE (Lector de la Tubería) ---
        close(fd[2]); // Cierra el extremo de escritura que no utiliza
        char buffer_lectura[3];
        std::cout << "  [PROCESO PADRE] Esperando lectura en el extremo fd (Lectura Bloqueante)...\n";
        read(fd, buffer_lectura, sizeof(buffer_lectura));
        std::cout << "  [PROCESO PADRE] Mensaje recibido del Hijo: \"" << buffer_lectura << "\"\n";
        close(fd);
        wait(NULL); // Recolecta proceso hijo
    }
}

int main() {
    std::cout << "=================================================================\n";
    std::cout << "   SIMULACIÓN DEL PROBLEMA PRODUCTOR-CONSUMIDOR CON SEMÁFOROS    \n";
    std::cout << "=================================================================\n\n";

    // Inicialización de Semáforos POSIX
    sem_init(&sem_huecos, 0, TAMANO_BUFFER); // Inicializado en N (5)
    sem_init(&sem_elementos, 0, 0);          // Inicializado en 0
    sem_init(&sem_mutex, 0, 1);              // Inicializado en 1 (Mutex binario)

    // Crear hilos concurrentes para Productor y Consumidor
    std::thread hilo_productor(productor);
    std::thread hilo_consumidor(consumidor);

    // Esperar finalización de hilos
    hilo_productor.join();
    hilo_consumidor.join();

    // Destruir semáforos
    sem_destroy(&sem_huecos);
    sem_destroy(&sem_elementos);
    sem_destroy(&sem_mutex);

    // Ejecutar demostración de tuberías entre procesos
    probar_tubería_pipe();

    return 0;
}
```

In [ ]:
%%writefile laboratorio_ipc_semaphoro.cpp
// ==============================================================================
// EJERCICIO PRÁCTICO: IPC Y SINCRONIZACIÓN CON SEMÁFOROS POSIX Y TUBERÍAS
// Lenguaje: C++17 (POSIX Threads & Semaphores)
// Asignatura: Sistemas Operativos - Unidad II: Administración de Procesos
// ==============================================================================

# include <iostream>
# include <thread>
# include <vector>
# include <chrono>
# include <semaphore.h>
# include <fcntl.h>
# include <unistd.h>
# include <sys/wait.h>

constexpr int TAMANO_BUFFER = 5;
constexpr int TOTAL_ITEMS = 10;

// Buffer circular compartido
int buffer[TAMANO_BUFFER];
int in_idx = 0;
int out_idx = 0;

// Semáforos POSIX
sem_t sem_huecos;    // Cuenta espacios libres en el buffer
sem_t sem_elementos; // Cuenta elementos disponibles para consumir
sem_t sem_mutex;     // Garantiza exclusión mutua (semáforo binario)

// Función del Proceso/Hilo Productor
void productor() {
    for (int i = 1; i <= TOTAL_ITEMS; ++i) {
        // 1. Producir dato
        int dato = i * 10;

        // 2. Esperar por un hueco libre
        sem_wait(&sem_huecos);

        // 3. Entrar a la región crítica (Mutex Down)
        sem_wait(&sem_mutex);

        // --- REGIÓN CRÍTICA ---
        buffer[in_idx] = dato;
        std::cout << "  [PRODUCTOR] Produjo e insertó item " << dato
                  << " en la posición " << in_idx << "\n";
        in_idx = (in_idx + 1) % TAMANO_BUFFER;
        // ----------------------

        // 4. Salir de la región crítica (Mutex Up)
        sem_post(&sem_mutex);

        // 5. Notificar que hay un nuevo elemento disponible
        sem_post(&sem_elementos);

        std::this_thread::sleep_for(std::chrono::milliseconds(150));
    }
}

// Función del Proceso/Hilo Consumidor
void consumidor() {
    for (int i = 1; i <= TOTAL_ITEMS; ++i) {
        // 1. Esperar a que exista al menos un elemento
        sem_wait(&sem_elementos);

        // 2. Entrar a la región crítica (Mutex Down)
        sem_wait(&sem_mutex);

        // --- REGIÓN CRÍTICA ---
        int dato = buffer[out_idx];
        std::cout << "    [CONSUMIDOR] Extrajo y consumió item " << dato
                  << " de la posición " << out_idx << "\n";
        out_idx = (out_idx + 1) % TAMANO_BUFFER;
        // ----------------------

        // 3. Salir de la región crítica (Mutex Up)
        sem_post(&sem_mutex);

        // 4. Notificar que se liberó un hueco
        sem_post(&sem_huecos);

        std::this_thread::sleep_for(std::chrono::milliseconds(300));
    }
}

// Función de demostración de Tubería Unidireccional (Pipe) entre procesos con fork()
void probar_tubería_pipe() {
    std::cout << "\n=================================================================\n";
    std::cout << "   DEMOSTRACIÓN DE COMUNICACIÓN POR TUBERÍA (PIPE EN LINUX)      \n";
    std::cout << "=================================================================\n";

    int fd[1]; // fd = lectura, fd[2] = escritura
    if (pipe(fd) == -1) {
        std::cerr << "Error al crear la tubería\n";
        return;
    }

    pid_t pid = fork();

    if (pid < 0) {
        std::cerr << "Error en fork()\n";
        return;
    }

    if (pid == 0) {
        // --- PROCESO HIJO (Escritor en la Tubería) ---
        close(fd); // Cierra el extremo de lectura que no utiliza
        std::string mensaje = "¡Saludos desde el Proceso Hijo mediante Pipe POSIX!";
        std::cout << "  [PROCESO HIJO] Enviando mensaje por el extremo de escritura fd[2]...\n";
        write(fd[2], mensaje.c_str(), mensaje.length() + 1);
        close(fd[2]); // Cierra escritura para enviar EOF
        _exit(0);
    } else {
        // --- PROCESO PADRE (Lector de la Tubería) ---
        close(fd[2]); // Cierra el extremo de escritura que no utiliza
        char buffer_lectura[3];
        std::cout << "  [PROCESO PADRE] Esperando lectura en el extremo fd (Lectura Bloqueante)...\n";
        read(fd, buffer_lectura, sizeof(buffer_lectura));
        std::cout << "  [PROCESO PADRE] Mensaje recibido del Hijo: \"" << buffer_lectura << "\"\n";
        close(fd);
        wait(NULL); // Recolecta proceso hijo
    }
}

int main() {
    std::cout << "=================================================================\n";
    std::cout << "   SIMULACIÓN DEL PROBLEMA PRODUCTOR-CONSUMIDOR CON SEMÁFOROS    \n";
    std::cout << "=================================================================\n\n";

    // Inicialización de Semáforos POSIX
    sem_init(&sem_huecos, 0, TAMANO_BUFFER); // Inicializado en N (5)
    sem_init(&sem_elementos, 0, 0);          // Inicializado en 0
    sem_init(&sem_mutex, 0, 1);              // Inicializado en 1 (Mutex binario)

    // Crear hilos concurrentes para Productor y Consumidor
    std::thread hilo_productor(productor);
    std::thread hilo_consumidor(consumidor);

    // Esperar finalización de hilos
    hilo_productor.join();
    hilo_consumidor.join();

    // Destruir semáforos
    sem_destroy(&sem_huecos);
    sem_destroy(&sem_elementos);
    sem_destroy(&sem_mutex);

    // Ejecutar demostración de tuberías entre procesos
    probar_tubería_pipe();

    return 0;
}

Writing laboratorio_ipc_semaphoro.cpp


### [CÓDIGO 4.3] Compilación y Ejecución del código.

```bash
# Compilar en C++17 con g++ habilitando bibliotecas de hilos y sincronización (-pthread)
!g++ -O3 -std=c++17 laboratorio_ipc_semaphoro.cpp -o laboratorio_ipc_semaphoro -pthread

# Ejecutar la simulación completa de IPC
!./laboratorio_ipc_semaphoro
```

In [ ]:
# Compilar en C++17 con g++ habilitando bibliotecas de hilos y sincronización (-pthread)
!g++ -O3 -std=c++17 laboratorio_ipc_semaphoro.cpp -o laboratorio_ipc_semaphoro -pthread

# Ejecutar la simulación completa de IPC
!./laboratorio_ipc_semaphoro

laboratorio_ipc_semaphoro.cpp: In function ‘void probar_tubería_pipe()’:
laboratorio_ipc_semaphoro.cpp:105:15: error: invalid conversion from ‘int*’ to ‘int’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-fpermissive-fpermissive]8;;]
  105 |         close(fd); // Cierra el extremo de lectura que no utiliza
      |               ^~
      |               |
      |               int*
In file included from laboratorio_ipc_semaphoro.cpp:13:
/usr/include/unistd.h:358:23: note:   initializing argument 1 of ‘int close(int)’
  358 | extern int close (int __fd);
      |                   ~~~~^~~~
laboratorio_ipc_semaphoro.cpp:116:14: error: invalid conversion from ‘int*’ to ‘int’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-fpermissive-fpermissive]8;;]
  116 |         read(fd, buffer_lectura, sizeof(buffer_lectura));
      |              ^~
      |              |
      |              int*
In file included from /usr/include/unistd.h:1217:
/usr/inc

## 6. Desafío Asíncrono / Entregable de la Unidad I

### Extensión y Análisis de la Tabla de Procesos y el BCP
A partir del laboratorio realizado en Google Colab sobre inspección de procesos en Linux y la estructura struct BCP en C++17, resuelve de manera individual los siguientes ítems modificando y extendiendo el código fuente base.

#### Instrucciones::
1. Análisis de Procesos del Sistema en Linux (/proc)
Contexto: El sistema de archivos virtual /proc contiene directorios nombrados con el PID de cada proceso activo, exponiendo archivos de control e información detallada del kernel.  
**Consigna:**  
> Ejecuta en una celda de Colab un comando o un pequeño script en Python/Bash que busque e imprima el uso de memoria actual y el estado de ejecución (campo State) de tres procesos distintos del sistema leyendo directamente sus archivos dentro de /proc/[PID]/status.  
Explica brevemente en un párrafo por qué el directorio /proc no ocupa espacio real en el disco duro de almacenamiento masivo.

2. Ampliación del Modelo de Datos del BCP en C++17
Contexto: Un sistema operativo real almacena más información de gestión en el Bloque de Control de Proceso, especialmente relacionada con la contabilidad de tiempo y la administración de archivos abiertos.  
**Consigna:**  
>Modifica el archivo bcp_struct.cpp para incorporar los siguientes campos dentro de la struct BCP:  
uint64_t tiempo_cpu_ms; (Tiempo total de CPU consumido en milisegundos).  
uint8_t archivos_abiertos; (Número de descriptores de archivos o sockets abiertos por el proceso).  
Actualiza el método mostrarBCP() para que despliegue formateados estos nuevos atributos en la ficha técnica de la consola.

3. Simulación de Transición de Estados
Contexto: Los procesos cambian dinámicamente de estado en función de los eventos del sistema y la planificación del kernel (Modelo de 5 Estados).  
**Consigna:**  
>Implementa una función en C++ dentro del programa que reciba por referencia un objeto BCP y un evento (por ejemplo, una interrupción por fin de E/S o una expiración de cuanto de tiempo/quantum), de modo que simule la transición de estado (por ejemplo, pasando de BLOQUEADO a LISTO) e imprima un mensaje descriptivo del cambio ocurrido.

### Gestión Avanzada y Ciclo de Vida de Procesos.  

#### Instrucciones:
A partir del laboratorio de jerarquía de procesos y simulación del BCP, resuelve de manera individual los siguientes ítems ampliando el código fuente de laboratorio_jerarquia_bcp.cpp.

1. Análisis de Procesos Zombie en Sistemas Linux Reales
Contexto: En los sistemas operativos tipo Unix, un proceso zombie ocurre cuando un hijo termina su ejecución, pero su padre aún no recolecta su estado de salida mediante la llamada wait().  
**Consigna:**  
> Investiga y redacta brevemente cómo se visualiza un proceso en estado zombie utilizando comandos de consola en Linux (por ejemplo, el indicador Z en ps aux o top).  
Responde en un párrafo de análisis: ¿Por qué el kernel de Linux mantiene el Bloque de Control de Proceso (BCP/Task Struct) en memoria para un proceso que ya ha terminado de ejecutarse y liberado su código en disco?  

2. Automatización de la Adopción de Huérfanos
Contexto: En el código base actual, la adopción de procesos huérfanos requiere invocar de manera explícita el método atender_huerfanos(). En un sistema operativo real, el kernel gestiona esto de forma automática cuando un proceso padre muere.  
**Consigna:**
> Modifica la lógica dentro de la clase SistemaOperativoSimulado para que, al ejecutar el método simular_exit(), el sistema detecte de forma automática si el proceso que termina tenía hijos activos y reasigne de inmediato su ppid al proceso init (PID 1), eliminando la necesidad de llamar al método de huérfanos por separado en el programa principal main.  

3. Extensión del Modelo de Estados (Estado Suspendido)
Contexto: Los modelos de procesos reales suelen incorporar más estados para manejar suspensiones temporales por intervención del usuario o del sistema (por ejemplo, mediante señales como SIGSTOP y SIGCONT).  
**Consigna:**  
> Agrega el estado SUSPENDIDO a la enumeración Estado.  
Implementa un nuevo método en el simulador llamado simular_suspension(int pid) que permita cambiar el estado de un proceso a SUSPENDIDO y guarde una traza en consola describiendo el cambio de contexto realizado.

### Concurrencia Avanzada y Robustez en IPC.  


A partir del laboratorio de mecanismos de IPC y sincronización, resuelve de manera individual los siguientes ítems ampliando y depurando el código fuente base.
#### Instrucciones:  

1. Auditoría de Recursos IPC en Linux.  
Contexto: Los recursos de IPC creados por los programas en sistemas Unix deben ser gestionados cuidadosamente por el kernel para evitar fugas de recursos (resource leaks).  
**Consigna:**  
> Ejecuta en una celda de Google Colab los comandos ipcs -m (para memoria compartida) e ipcs -s (para semáforos del sistema) y captura su salida.  
Explica brevemente en un párrafo qué representan las columnas clave (key), identificador (shmid/semid) y propietario (owner) que muestra el sistema operativo.

2. Escalabilidad a Múltiples Productores y Consumidores.  
Contexto: En escenarios reales de sistemas operativos, múltiples hilos o procesos intentan acceder concurrentemente a los recursos compartidos.  
**Consigna:**
> Modifica la función main() del código base para instanciar dos hilos productores y dos hilos consumidores de forma simultánea. Asegúrate de verificar que el semáforo de exclusión mutua (sem_mutex) y los contadores de huecos/elementos previenen eficazmente las condiciones de carrera y la corrupción del buffer circular.

3. Depuración y Análisis de Descriptores en Tuberías (Pipe).  
Contexto: La correcta gestión de los descriptores de archivos (file descriptors) es vital en las tuberías para evitar bloqueos permanentes (deadlocks) por lecturas colgadas en la espera de un fin de archivo (EOF).  
**Consigna:**  
> Analiza la función probar_tubería_pipe(). Identifica por qué el manejo de los descriptores de archivo requiere un arreglo de tamaño 2 (int fd[2]) para separar adecuadamente el canal de lectura (fd) del canal de escritura (fd), y documenta qué ocurre si el proceso padre olvida cerrar el extremo de escritura antes de realizar la operación de lectura bloqueante.
---

### Formato de Entrega / Portafolio de Evidencias
1. Guarda este cuaderno con todas las salidas ejecutadas incluyendo las respuesta a las preguntas (`Archivo` -> `Guardar una copia en Drive`).
2. Sube los cambios a tu repositorio de GitHub de la materia bajo el nombre `JUAN PEREZ Practica_Sistemas Operativos U2.ipynb` usando la integración directa (`Archivo` -> `Guardar una copia en GitHub`).
3. Adjunta en tu entregable de plataforma Moodle, el enlace de GitHub.